<a href="https://colab.research.google.com/github/Annaa74/Google-colab-models/blob/main/AI_Powered_Resume_Screening_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import streamlit as st
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import pickle
import io

# Download required NLTK data
try:
    nltk.data.find('tokenizers/punkt')
    nltk.data.find('corpora/stopwords')
    nltk.data.find('corpora/wordnet')
except LookupError:
    nltk.download('punkt')
    nltk.download('stopwords')
    nltk.download('wordnet')

class ResumeScreener:
    def __init__(self):
        self.vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
        self.model = None
        self.lemmatizer = WordNetLemmatizer()
        self.stop_words = set(stopwords.words('english'))

    def preprocess_text(self, text):
        """Clean and preprocess resume text"""
        # Convert to lowercase
        text = text.lower()

        # Remove special characters and digits
        text = re.sub(r'[^a-zA-Z\s]', '', text)

        # Remove extra whitespace
        text = re.sub(r'\s+', ' ', text).strip()

        # Tokenize
        tokens = word_tokenize(text)

        # Remove stopwords and lemmatize
        tokens = [self.lemmatizer.lemmatize(token) for token in tokens
                 if token not in self.stop_words and len(token) > 2]

        return ' '.join(tokens)

    def extract_features(self, resume_text):
        """Extract additional features from resume"""
        features = {}

        # Count of different sections
        features['education_mentions'] = len(re.findall(r'\b(education|degree|university|college|bachelor|master|phd)\b', resume_text.lower()))
        features['experience_mentions'] = len(re.findall(r'\b(experience|worked|job|position|role|company)\b', resume_text.lower()))
        features['skills_mentions'] = len(re.findall(r'\b(skills|python|java|sql|machine learning|data science|programming)\b', resume_text.lower()))
        features['projects_mentions'] = len(re.findall(r'\b(project|developed|built|created|implemented)\b', resume_text.lower()))

        # Text length features
        features['text_length'] = len(resume_text)
        features['word_count'] = len(resume_text.split())

        return features

    def train_model(self, resumes, labels, model_type='random_forest'):
        """Train the resume screening model"""
        # Preprocess texts
        processed_texts = [self.preprocess_text(text) for text in resumes]

        # Vectorize texts
        X_text = self.vectorizer.fit_transform(processed_texts)

        # Extract additional features
        additional_features = []
        for text in resumes:
            features = self.extract_features(text)
            additional_features.append(list(features.values()))

        # Combine text features with additional features
        X_additional = np.array(additional_features)
        X = np.hstack([X_text.toarray(), X_additional])

        # Split data
        X_train, X_test, y_train, y_test = train_test_split(X, labels, test_size=0.2, random_state=42)

        # Train model
        if model_type == 'random_forest':
            self.model = RandomForestClassifier(n_estimators=100, random_state=42)
        else:
            self.model = LogisticRegression(random_state=42, max_iter=1000)

        self.model.fit(X_train, y_train)

        # Evaluate
        y_pred = self.model.predict(X_test)
        accuracy = accuracy_score(y_test, y_pred)

        return accuracy, classification_report(y_test, y_pred)

    def predict_resume(self, resume_text):
        """Predict if a resume should be selected"""
        if self.model is None:
            return None, None

        # Preprocess
        processed_text = self.preprocess_text(resume_text)

        # Vectorize
        X_text = self.vectorizer.transform([processed_text])

        # Extract additional features
        features = self.extract_features(resume_text)
        X_additional = np.array([list(features.values())])

        # Combine features
        X = np.hstack([X_text.toarray(), X_additional])

        # Predict
        prediction = self.model.predict(X)[0]
        probability = self.model.predict_proba(X)[0]

        return prediction, max(probability)

def create_sample_data():
    """Create sample resume data for demonstration"""
    sample_resumes = [
        "John Doe. Software Engineer with 5 years experience in Python, Java, and SQL. Bachelor's degree in Computer Science from MIT. Worked at Google and Microsoft. Led multiple machine learning projects including recommendation systems and data analytics platforms.",

        "Jane Smith. Data Scientist with PhD in Statistics from Stanford. 3 years experience at Facebook working on deep learning models. Expert in Python, R, TensorFlow, and PyTorch. Published 10 research papers in top-tier conferences.",

        "Mike Johnson. Recent graduate with Bachelor's in Information Technology. Basic knowledge of HTML, CSS, and JavaScript. Completed internship at local startup. Looking for entry-level position.",

        "Sarah Wilson. Senior Software Developer with 8 years experience. Master's degree in Computer Science. Expert in distributed systems, microservices, and cloud computing. Led teams of 10+ developers at Amazon.",

        "Tom Brown. Self-taught programmer with 2 years freelance experience. High school diploma. Knowledge of WordPress, basic PHP, and MySQL. Built several small websites for local businesses.",

        "Lisa Davis. Machine Learning Engineer with 4 years experience at Tesla. Master's in AI from Carnegie Mellon. Expert in computer vision, autonomous systems, and deep learning. Led self-driving car perception team.",

        "Alex Chen. Full-stack developer with 6 years experience. Bachelor's in Software Engineering. Proficient in React, Node.js, MongoDB, and AWS. Built scalable web applications serving millions of users.",

        "Emma Garcia. Data Analyst with 2 years experience. Bachelor's in Mathematics. Skilled in Excel, SQL, and Tableau. Experience with statistical analysis and business intelligence reporting."
    ]

    # Labels: 1 for selected, 0 for not selected
    labels = [1, 1, 0, 1, 0, 1, 1, 0]  # Based on experience and qualifications

    return sample_resumes, labels

def main():
    st.set_page_config(page_title="AI Resume Screener", page_icon="📄", layout="wide")

    st.title("🤖 AI-Powered Resume Screening System")
    st.markdown("---")

    # Initialize the screener
    if 'screener' not in st.session_state:
        st.session_state.screener = ResumeScreener()
        st.session_state.model_trained = False

    # Sidebar
    st.sidebar.title("Model Configuration")

    # Model training section
    st.sidebar.subheader("1. Train Model")

    if st.sidebar.button("Load Sample Data & Train"):
        with st.spinner("Training model..."):
            sample_resumes, labels = create_sample_data()

            model_type = st.sidebar.selectbox("Choose Model", ["random_forest", "logistic_regression"])

            accuracy, report = st.session_state.screener.train_model(
                sample_resumes, labels, model_type
            )

            st.session_state.model_trained = True
            st.session_state.accuracy = accuracy
            st.session_state.report = report

            st.sidebar.success(f"Model trained! Accuracy: {accuracy:.2%}")

    # File upload section
    st.sidebar.subheader("2. Upload Custom Data")
    uploaded_file = st.sidebar.file_uploader(
        "Upload CSV with 'resume' and 'label' columns",
        type=['csv']
    )

    if uploaded_file is not None:
        df = pd.read_csv(uploaded_file)
        if 'resume' in df.columns and 'label' in df.columns:
            if st.sidebar.button("Train on Uploaded Data"):
                with st.spinner("Training on uploaded data..."):
                    model_type = st.sidebar.selectbox("Choose Model", ["random_forest", "logistic_regression"], key="upload_model")
                    accuracy, report = st.session_state.screener.train_model(
                        df['resume'].tolist(), df['label'].tolist(), model_type
                    )
                    st.session_state.model_trained = True
                    st.session_state.accuracy = accuracy
                    st.session_state.report = report
                    st.sidebar.success(f"Model trained! Accuracy: {accuracy:.2%}")
        else:
            st.sidebar.error("CSV must contain 'resume' and 'label' columns")

    # Main content
    col1, col2 = st.columns([2, 1])

    with col1:
        st.subheader("📝 Resume Input")

        # Text input methods
        input_method = st.radio("Choose input method:", ["Paste Text", "Upload File"])

        resume_text = ""

        if input_method == "Paste Text":
            resume_text = st.text_area(
                "Paste resume text here:",
                height=300,
                placeholder="Enter the resume text you want to analyze..."
            )
        else:
            uploaded_resume = st.file_uploader(
                "Upload resume file",
                type=['txt', 'pdf'],
                help="Currently supports TXT files. PDF support requires additional libraries."
            )

            if uploaded_resume is not None:
                if uploaded_resume.type == "text/plain":
                    resume_text = str(uploaded_resume.read(), "utf-8")
                    st.success("Resume uploaded successfully!")
                else:
                    st.warning("PDF support not implemented. Please use TXT files or paste text directly.")

        # Prediction section
        if resume_text and st.session_state.model_trained:
            if st.button("🔍 Analyze Resume", type="primary"):
                with st.spinner("Analyzing resume..."):
                    prediction, confidence = st.session_state.screener.predict_resume(resume_text)

                    st.markdown("---")
                    st.subheader("📊 Analysis Results")

                    if prediction == 1:
                        st.success(f"✅ **RECOMMENDED** (Confidence: {confidence:.2%})")
                        st.balloons()
                    else:
                        st.error(f"❌ **NOT RECOMMENDED** (Confidence: {confidence:.2%})")

                    # Feature analysis
                    features = st.session_state.screener.extract_features(resume_text)

                    st.subheader("📈 Feature Analysis")
                    feature_col1, feature_col2 = st.columns(2)

                    with feature_col1:
                        st.metric("Education Mentions", features['education_mentions'])
                        st.metric("Experience Mentions", features['experience_mentions'])
                        st.metric("Skills Mentions", features['skills_mentions'])

                    with feature_col2:
                        st.metric("Projects Mentions", features['projects_mentions'])
                        st.metric("Word Count", features['word_count'])
                        st.metric("Text Length", features['text_length'])

        elif resume_text and not st.session_state.model_trained:
            st.warning("⚠️ Please train the model first using the sidebar options.")

    with col2:
        st.subheader("📈 Model Performance")

        if st.session_state.model_trained:
            st.success(f"Model Accuracy: {st.session_state.accuracy:.2%}")

            st.subheader("📋 Classification Report")
            st.text(st.session_state.report)

            # Sample data preview
            st.subheader("📋 Sample Training Data")
            sample_resumes, labels = create_sample_data()
            sample_df = pd.DataFrame({
                'Resume Preview': [resume[:100] + "..." for resume in sample_resumes],
                'Label': labels
            })
            st.dataframe(sample_df, use_container_width=True)

        else:
            st.info("No model trained yet. Use the sidebar to train a model.")

            st.subheader("🎯 About This Tool")
            st.markdown("""
            This AI resume screener uses:

            **Features:**
            - Text preprocessing & cleaning
            - TF-IDF vectorization
            - Custom feature extraction
            - Machine learning classification

            **Models Available:**
            - Random Forest
            - Logistic Regression

            **Training Options:**
            - Sample data (8 resumes)
            - Upload custom CSV data
            """)

if __name__ == "__main__":
    main()

ModuleNotFoundError: No module named 'streamlit'